In [1]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

## Inspect table schema
Run this cell first to confirm available columns before querying.

In [2]:
conn = duckdb.connect(r"/scratch/duckdb-database/main_survey_database.duckdb")

table_name = 'PCL_dens_percell_3D'

schema = conn.execute(f"DESCRIBE {table_name}").fetchdf()
print("Table schema:")
display(schema)

cell_types = conn.execute(f"SELECT DISTINCT cell_type FROM {table_name} ORDER BY cell_type").fetchdf()
regions    = conn.execute(f"SELECT DISTINCT brain_region FROM {table_name} ORDER BY brain_region").fetchdf()
print("\nCell types:", cell_types['cell_type'].tolist())
print("Brain regions:", regions['brain_region'].tolist())

conn.close()

Table schema:


,column_name,column_type,null,key,default,extra
0,area/um3,DOUBLE,YES,None,None,None
1,z_centre,BIGINT,YES,None,None,None
2,x_centre,BIGINT,YES,None,None,None
3,y_centre,BIGINT,YES,None,None,None
4,puncta_cell_likelihood,DOUBLE,YES,None,None,None
5,n_puncta_in_cell,BIGINT,YES,None,None,None
6,image_filename,VARCHAR,YES,None,None,None
7,id,VARCHAR,YES,None,None,None
8,donorid,VARCHAR,YES,None,None,None
9,incelldens,DOUBLE,YES,None,None,None



Cell types: ['microglia', 'neurons']
Brain regions: ['caudate', 'cingulate', 'frontal', 'parahippocampal', 'parietal', 'putamen', 'substantiaNigra', 'temporal']


## Query per-cell microglia data

### Column definitions and units

| Column | Source | Unit | Description |
|---|---|---|---|
| `cell_volume_um3` | `area/um3` | µm³ | Measured volume of the individual cell |
| `incelldens` | `incelldens` | **oligomers / µm³** | Raw in-cell αSyn oligomer density (key parameter) |
| `n_puncta_in_cell` | `n_puncta_in_cell` | oligomers | Total αSyn oligomer count detected inside the cell |

> **Note on `incelldens`**: this is the primary density metric (αSyn oligomers per µm³ of cell volume). It is preserved as-is from the database under its original name so it can be unambiguously traced back to the source data.

Filters applied (matching the standard microglia analysis):
- `percentile = 0` (no intensity threshold)
- `100 ≤ area/um3 ≤ 500 µm³` (cell volume QC)

In [3]:
conn = duckdb.connect(r"/scratch/duckdb-database/main_survey_database.duckdb")

table_name       = 'PCL_dens_percell_3D'
cell_type_filter  = 'microglia'
percentile_filter = 0
vol_min = 100
vol_max = 500

# incelldens       unit: oligomers / µm³  (key density parameter)
# n_puncta_in_cell unit: oligomers        (total count per cell, from DB)
query = f"""
SELECT
    donorid,
    disease,
    brain_region,
    cell_type,
    "area/um3"       AS cell_volume_um3,
    incelldens,
    n_puncta_in_cell
FROM {table_name}
WHERE cell_type = '{cell_type_filter}'
    AND percentile = {percentile_filter}
    AND "area/um3" >= {vol_min}
    AND "area/um3" <= {vol_max}
ORDER BY donorid, brain_region
"""

per_cell = conn.execute(query).fetchdf()
conn.close()

print(f"Total cells retrieved: {len(per_cell):,}")
print(f"Donors: {per_cell['donorid'].nunique()}")
print(f"Brain regions: {sorted(per_cell['brain_region'].unique())}")
print(f"Disease groups: {sorted(per_cell['disease'].unique())}")
per_cell.head()

Total cells retrieved: 13,214
Donors: 27
Brain regions: ['caudate', 'cingulate', 'frontal', 'parahippocampal', 'parietal', 'putamen', 'substantiaNigra', 'temporal']
Disease groups: ['HC', 'PD']


,donorid,disease,brain_region,cell_type,cell_volume_um3,incelldens,n_puncta_in_cell
0,C030,HC,cingulate,microglia,178.11200,0.011229,2
1,C030,HC,cingulate,microglia,399.95945,0.027503,11
2,C030,HC,cingulate,microglia,184.24065,0.016283,3
3,C030,HC,cingulate,microglia,101.57345,0.068916,7
4,C030,HC,cingulate,microglia,386.74020,0.033614,13


## Compute summary statistics

Two levels of aggregation (patient → region hierarchy):
1. **Patient × Region**: per donor, per brain region
2. **Patient** (all regions combined)

For each level: n cells, mean/median/SD of `n_puncta_in_cell` (oligomers) and `incelldens` (oligomers / µm³).

In [4]:
# incelldens       unit: oligomers / µm³
# n_puncta_in_cell unit: oligomers

q25 = lambda x: x.quantile(0.25)
q75 = lambda x: x.quantile(0.75)
q25.__name__ = 'q25'
q75.__name__ = 'q75'

# --- Summary: per patient, per brain region ---
summary_patient_region = (
    per_cell
    .groupby(['donorid', 'disease', 'brain_region'], sort=True)
    .agg(
        n_microglia                      = ('n_puncta_in_cell', 'count'),
        mean_n_puncta_in_cell            = ('n_puncta_in_cell', 'mean'),
        median_n_puncta_in_cell          = ('n_puncta_in_cell', 'median'),
        q25_n_puncta_in_cell             = ('n_puncta_in_cell', q25),
        q75_n_puncta_in_cell             = ('n_puncta_in_cell', q75),
        sd_n_puncta_in_cell              = ('n_puncta_in_cell', 'std'),
        mean_cell_volume_um3             = ('cell_volume_um3', 'mean'),
        median_cell_volume_um3           = ('cell_volume_um3', 'median'),
        # incelldens: oligomers / µm³
        mean_incelldens_oligos_per_um3   = ('incelldens', 'mean'),
        median_incelldens_oligos_per_um3 = ('incelldens', 'median'),
        q25_incelldens_oligos_per_um3    = ('incelldens', q25),
        q75_incelldens_oligos_per_um3    = ('incelldens', q75),
        sd_incelldens_oligos_per_um3     = ('incelldens', 'std'),
    )
    .reset_index()
)

float_cols = summary_patient_region.select_dtypes(include='float').columns
summary_patient_region[float_cols] = summary_patient_region[float_cols].round(4)

print(f"Patient × Region rows: {len(summary_patient_region)}")
display(summary_patient_region.head(10))

# --- Summary: per patient (all regions collapsed) ---
summary_patient = (
    per_cell
    .groupby(['donorid', 'disease'], sort=True)
    .agg(
        n_microglia                      = ('n_puncta_in_cell', 'count'),
        n_regions                        = ('brain_region', 'nunique'),
        mean_n_puncta_in_cell            = ('n_puncta_in_cell', 'mean'),
        median_n_puncta_in_cell          = ('n_puncta_in_cell', 'median'),
        q25_n_puncta_in_cell             = ('n_puncta_in_cell', q25),
        q75_n_puncta_in_cell             = ('n_puncta_in_cell', q75),
        sd_n_puncta_in_cell              = ('n_puncta_in_cell', 'std'),
        mean_cell_volume_um3             = ('cell_volume_um3', 'mean'),
        median_cell_volume_um3           = ('cell_volume_um3', 'median'),
        # incelldens: oligomers / µm³
        mean_incelldens_oligos_per_um3   = ('incelldens', 'mean'),
        median_incelldens_oligos_per_um3 = ('incelldens', 'median'),
        q25_incelldens_oligos_per_um3    = ('incelldens', q25),
        q75_incelldens_oligos_per_um3    = ('incelldens', q75),
        sd_incelldens_oligos_per_um3     = ('incelldens', 'std'),
    )
    .reset_index()
)

float_cols = summary_patient.select_dtypes(include='float').columns
summary_patient[float_cols] = summary_patient[float_cols].round(4)

print(f"\nPatient-level rows: {len(summary_patient)}")
display(summary_patient)

Patient × Region rows: 205


,donorid,disease,brain_region,n_microglia,mean_n_puncta_in_cell,median_n_puncta_in_cell,q25_n_puncta_in_cell,q75_n_puncta_in_cell,sd_n_puncta_in_cell,mean_cell_volume_um3,median_cell_volume_um3,mean_incelldens_oligos_per_um3,median_incelldens_oligos_per_um3,q25_incelldens_oligos_per_um3,q75_incelldens_oligos_per_um3,sd_incelldens_oligos_per_um3
0,C030,HC,cingulate,25,4.5200,3.0,2.00,7.00,3.5014,217.3387,182.6192,0.0236,0.0164,0.0110,0.0294,0.0200
1,C030,HC,frontal,32,21.4375,18.0,8.50,25.25,18.9378,179.0949,153.5823,0.1243,0.1003,0.0570,0.1909,0.0957
2,C030,HC,parietal,55,34.0182,25.0,13.00,49.00,28.4524,186.4114,146.9303,0.1846,0.1427,0.0895,0.2450,0.1412
3,C030,HC,putamen,42,22.6905,18.5,11.25,32.75,14.6662,163.6156,129.4609,0.1500,0.1230,0.0791,0.1675,0.1052
4,C030,HC,substantiaNigra,144,16.4792,12.0,7.00,22.00,13.4549,195.8049,161.7256,0.0870,0.0740,0.0416,0.1179,0.0591
5,C030,HC,temporal,21,31.8095,28.0,17.00,44.00,19.9815,156.4697,140.0817,0.1994,0.1990,0.1207,0.2884,0.1005
6,C030,PD,substantiaNigra,93,14.3871,11.0,5.00,19.00,12.8156,203.1169,170.8036,0.0707,0.0572,0.0314,0.0888,0.0551
7,C043,HC,caudate,169,12.8994,10.0,6.00,18.00,10.6008,209.7016,177.1440,0.0636,0.0576,0.0305,0.0854,0.0416
8,C043,HC,cingulate,48,4.5000,3.0,1.00,6.00,4.2426,229.7977,204.8651,0.0210,0.0163,0.0089,0.0241,0.0218
9,C043,HC,frontal,56,39.6429,33.5,17.75,53.00,28.9441,177.5570,151.9488,0.2278,0.2214,0.1086,0.2941,0.1353



Patient-level rows: 29


,donorid,disease,n_microglia,n_regions,mean_n_puncta_in_cell,median_n_puncta_in_cell,q25_n_puncta_in_cell,q75_n_puncta_in_cell,sd_n_puncta_in_cell,mean_cell_volume_um3,median_cell_volume_um3,mean_incelldens_oligos_per_um3,median_incelldens_oligos_per_um3,q25_incelldens_oligos_per_um3,q75_incelldens_oligos_per_um3,sd_incelldens_oligos_per_um3
0,C030,HC,319,6,20.8903,14.0,8.0,27.5,19.3811,187.3692,153.6821,0.1183,0.0953,0.0463,0.1556,0.1021
1,C030,PD,93,1,14.3871,11.0,5.0,19.0,12.8156,203.1169,170.8036,0.0707,0.0572,0.0314,0.0888,0.0551
2,C043,HC,597,7,23.5745,16.0,7.0,31.0,25.5832,195.5854,160.7243,0.1316,0.0847,0.0394,0.1685,0.1429
3,C046,HC,569,8,13.6784,9.0,4.0,18.0,14.2682,186.3056,163.3560,0.0767,0.0512,0.0265,0.1003,0.0732
4,C073,HC,271,7,26.4317,18.0,6.5,38.0,26.8756,167.8661,145.2000,0.1589,0.1128,0.0422,0.2389,0.1471
5,C074,HC,337,7,35.6766,28.0,13.0,48.0,32.1811,179.2359,156.4046,0.2032,0.1663,0.0781,0.2755,0.1588
6,C075,HC,431,8,25.7981,15.0,6.0,36.0,33.2650,201.5386,171.0396,0.1274,0.0791,0.0335,0.1791,0.1305
7,C076,HC,364,7,18.4863,13.0,5.0,27.0,17.5167,180.9571,155.7270,0.1075,0.0808,0.0348,0.1476,0.1033
8,PD0022,PD,799,6,17.7309,13.0,6.0,24.0,16.4307,198.5423,169.0914,0.0938,0.0755,0.0347,0.1342,0.0770
9,PD0086,PD,532,8,28.6917,20.0,11.0,38.0,25.2406,189.8603,159.7896,0.1535,0.1242,0.0655,0.2087,0.1202


## Export

- **Per-cell data → Parquet** (`.parquet`): one row per individual microglial cell; saved as Parquet because the row count would be unwieldy in Excel, and Parquet handles mixed string/numeric columns efficiently without pickling
- **Summary statistics → Excel** (`.xlsx`): two sheets with mean, median, Q25, Q75, SD aggregated at patient×region and patient level

In [5]:
output_dir = Path(r"/scratch/sycamore-asap/ASAP_Plots")

# --- Per-cell data: Parquet (handles string columns natively, no pickling) ---
parquet_path = output_dir / "microglia_per_cell_data.parquet"
per_cell.to_parquet(parquet_path, index=False, compression='snappy')
print(f"Saved per-cell Parquet: {parquet_path}  ({len(per_cell):,} rows)")
print(f"  n_microglia total: {len(per_cell):,}")

# --- Summary statistics: Excel ---
xlsx_path = output_dir / "microglia_oligomer_in_cell_statistics.xlsx"
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    summary_patient_region.to_excel(writer, sheet_name='Summary_Patient_Region', index=False)
    summary_patient.to_excel(writer,        sheet_name='Summary_Patient',        index=False)

print(f"\nSaved summary Excel:  {xlsx_path}")
print(f"  Sheet 'Summary_Patient_Region': {len(summary_patient_region):,} rows")
print(f"  Sheet 'Summary_Patient':        {len(summary_patient):,} rows")

Saved per-cell Parquet: /scratch/sycamore-asap/ASAP_Plots/microglia_per_cell_data.parquet  (13,214 rows)
  n_microglia total: 13,214

Saved summary Excel:  /scratch/sycamore-asap/ASAP_Plots/microglia_oligomer_in_cell_statistics.xlsx
  Sheet 'Summary_Patient_Region': 205 rows
  Sheet 'Summary_Patient':        29 rows
